# Tennis Event Benchmark

Use this notebook to build a small reviewed benchmark for event extraction.

Workflow:
1. pick a sample clip
2. render or reuse the event-annotated source and bird's-eye videos
3. edit the benchmark label JSON
4. save it and score the current predictions against those labels

Label files are stored in `data/benchmark/labels/`.


In [1]:
from __future__ import annotations

import html
import json
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Video, display

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tennis_tracker.benchmark import (
    score_event_predictions_data,
    write_benchmark_label_template,
)
from tennis_tracker.birdseye import render_birdseye_video
from tennis_tracker.events import (
    ensure_match_events_for_video,
    render_event_annotation_video,
    slugify_video_name,
    summarize_match_events,
)
from tennis_tracker.pose import extract_player_poses

SAMPLES_DIR = ROOT / "data" / "samples"
SAMPLE_VIDEOS = sorted(SAMPLES_DIR.glob("*.mp4"))
assert SAMPLE_VIDEOS, f"No sample videos found in {SAMPLES_DIR}"

DEFAULT_VIDEO = next((p for p in SAMPLE_VIDEOS if p.name == "broadcast_2s.mp4"), SAMPLE_VIDEOS[0])
WEIGHTS_PATH = ROOT / "models" / "tracknet_weights.pth"
POSE_MODEL = ROOT / "models" / "yolo11n-pose.pt"
if not POSE_MODEL.exists():
    POSE_MODEL = Path("yolo11n-pose.pt")

EVENTS_OUT_DIR = ROOT / "out" / "events"
EVENTS_OUT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR = ROOT / "data" / "benchmark" / "labels"
LABEL_DIR.mkdir(parents=True, exist_ok=True)

selector = widgets.Dropdown(
    options=[(video.name, str(video)) for video in SAMPLE_VIDEOS],
    value=str(DEFAULT_VIDEO),
    description="Video:",
    layout=widgets.Layout(width="430px"),
)
force_rerun = widgets.Checkbox(value=False, description="Force rerun")
render_button = widgets.Button(description="Load benchmark clip", button_style="primary")
save_button = widgets.Button(description="Save labels")
score_button = widgets.Button(description="Score predictions")
preview_output = widgets.Output()
score_output = widgets.Output()
label_editor = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="420px"),
    placeholder="Benchmark label JSON will appear here.",
)
current_state: dict[str, Path] = {}


def build_video_panel(video_path: Path, title: str) -> widgets.HTML:
    video = Video(
        filename=str(video_path),
        embed=True,
        width=560,
        html_attributes="controls autoplay loop muted playsinline",
    )
    video_html = video._repr_html_() or ""
    return widgets.HTML(
        value=(
            "<div style='width:100%'>"
            f"<div style='font-weight:600; margin-bottom:6px;'>{title}</div>"
            f"{video_html}"
            "</div>"
        ),
        layout=widgets.Layout(width="50%"),
    )


def is_stale(output_path: Path, *input_paths: Path) -> bool:
    if not output_path.exists():
        return True
    output_mtime = output_path.stat().st_mtime
    return any(path.exists() and path.stat().st_mtime > output_mtime for path in input_paths)


def label_path_for(video_path: Path) -> Path:
    return LABEL_DIR / f"{slugify_video_name(video_path.name)}_labels.json"


def ensure_visual_artifacts(video_path: Path, *, force: bool = False) -> dict[str, Path]:
    slug = slugify_video_name(video_path.name)
    artifacts = ensure_match_events_for_video(
        video_path=video_path,
        output_dir=EVENTS_OUT_DIR,
        weights_path=WEIGHTS_PATH,
        force=force,
    )

    pose_json_path = EVENTS_OUT_DIR / f"{slug}_pose.json"
    birdseye_video_path = EVENTS_OUT_DIR / f"{slug}_birdseye.mp4"
    source_events_video_path = EVENTS_OUT_DIR / f"{slug}_events_source.mp4"
    birdseye_events_video_path = EVENTS_OUT_DIR / f"{slug}_events_birdseye.mp4"

    if force or is_stale(pose_json_path, artifacts["tracking_json"]):
        extract_player_poses(
            video_path=video_path,
            tracking_json_path=artifacts["tracking_json"],
            output_json_path=pose_json_path,
            model_name=POSE_MODEL,
            imgsz=256,
            batch_size=16,
        )

    if force or is_stale(birdseye_video_path, artifacts["tracking_json"], pose_json_path):
        render_birdseye_video(
            tracking_json_path=artifacts["tracking_json"],
            output_video_path=birdseye_video_path,
            pose_json_path=pose_json_path,
        )

    if force or is_stale(source_events_video_path, artifacts["tracking_video"], artifacts["events_json"]):
        render_event_annotation_video(
            input_video_path=artifacts["tracking_video"],
            events_json_path=artifacts["events_json"],
            output_video_path=source_events_video_path,
        )

    if force or is_stale(birdseye_events_video_path, birdseye_video_path, artifacts["events_json"]):
        render_event_annotation_video(
            input_video_path=birdseye_video_path,
            events_json_path=artifacts["events_json"],
            output_video_path=birdseye_events_video_path,
        )

    artifacts["source_events_video"] = source_events_video_path
    artifacts["birdseye_events_video"] = birdseye_events_video_path
    return artifacts


def refresh_benchmark(video_path: Path, *, force: bool = False) -> None:
    artifacts = ensure_visual_artifacts(video_path, force=force)
    label_path = label_path_for(video_path)
    if not label_path.exists():
        write_benchmark_label_template(
            prediction_json_path=artifacts["events_json"],
            output_label_path=label_path,
        )

    current_state.clear()
    current_state.update(artifacts)
    current_state["label_json"] = label_path
    label_editor.value = label_path.read_text()

    summary = summarize_match_events(artifacts["events_json"])
    summary["label_json"] = str(label_path.relative_to(ROOT))

    preview_output.clear_output(wait=True)
    with preview_output:
        print(summary)
        display(
            widgets.HBox(
                [
                    build_video_panel(artifacts["source_events_video"], "Tracked source + events"),
                    build_video_panel(artifacts["birdseye_events_video"], "Bird's-eye + events"),
                ],
                layout=widgets.Layout(align_items="flex-start"),
            )
        )
        display(
            widgets.HTML(
                value=(
                    "<div style='margin-top:12px;'>Review the current predictions in the videos, then edit the benchmark label JSON below.</div>"
                )
            )
        )

    score_output.clear_output(wait=True)
    with score_output:
        print("Benchmark labels loaded. Save or score after editing the JSON.")


def save_labels() -> None:
    if not current_state:
        raise RuntimeError("Load a benchmark clip first.")
    label_data = json.loads(label_editor.value)
    current_state["label_json"].write_text(json.dumps(label_data, indent=2))


def score_labels() -> dict[str, object]:
    if not current_state:
        raise RuntimeError("Load a benchmark clip first.")
    prediction_data = json.loads(current_state["events_json"].read_text())
    label_data = json.loads(label_editor.value)
    return score_event_predictions_data(
        prediction_data=prediction_data,
        label_data=label_data,
    )


In [2]:
def on_render_clicked(_: widgets.Button) -> None:
    refresh_benchmark(Path(selector.value), force=force_rerun.value)


def on_save_clicked(_: widgets.Button) -> None:
    score_output.clear_output(wait=True)
    with score_output:
        try:
            save_labels()
            print(f"Saved {current_state['label_json'].relative_to(ROOT)}")
        except Exception as exc:
            print(f"Save failed: {exc}")


def on_score_clicked(_: widgets.Button) -> None:
    score_output.clear_output(wait=True)
    with score_output:
        try:
            results = score_labels()
            print(json.dumps(results, indent=2))
        except Exception as exc:
            print(f"Scoring failed: {exc}")


render_button.on_click(on_render_clicked)
save_button.on_click(on_save_clicked)
score_button.on_click(on_score_clicked)

controls = widgets.HBox([selector, force_rerun, render_button, save_button, score_button])
display(widgets.VBox([controls, preview_output, label_editor, score_output]))
refresh_benchmark(Path(selector.value), force=False)
